# Aula 7 - Otimização de hiperparâmetros
## Grid search (slides 9–12)


In [2]:
import numpy as np
from si.data.dataset import Dataset
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.layers import DenseLayer
from si.neural_networks.activation import ReLUActivation, SigmoidActivation
from si.neural_networks.optimizers import SGD
from si.neural_networks.losses import BinaryCrossEntropy
from si.metrics.accuracy import accuracy
from si.model_selection.cross_validation import k_fold_cross_validation
from si.model_selection.grid_search import grid_search_cv

In [3]:
def build_network(learning_rate, n_hidden):
    """
    Função construtora (slide 10): devolve uma rede nova, por treinar, com estes hiperparâmetros.
    O grid search chama-a uma vez por fold, por isso nunca se reaproveitam pesos.
    """
    # a mesma seed em todas as chamadas: as redes começam com os mesmos pesos iniciais e usam os
    # mesmos mini-batches, por isso as diferenças nos scores vêm só dos hiperparâmetros
    np.random.seed(0)
    net = NeuralNetwork(epochs=100, batch_size=16, optimizer=SGD,
                        learning_rate=learning_rate, momentum=0.9,
                        loss=BinaryCrossEntropy, metric=accuracy)
    # 2 features --> n_hidden neurónios (ReLU) --> 1 saída (sigmoide: probabilidade de estar dentro do círculo)
    net.add(DenseLayer(n_hidden, (2,)))
    net.add(ReLUActivation())
    net.add(DenseLayer(1))
    net.add(SigmoidActivation())
    return net

In [4]:
# dados (slide 12): 400 pontos no quadrado [-2, 2] x [-2, 2]
# classe 1 se o ponto estiver dentro do círculo x1² + x2² < 1.5
rng = np.random.default_rng(42)
X = rng.uniform(-2, 2, size=(400, 2))
y = (X[:, 0]**2 + X[:, 1]**2 < 1.5).astype(float)
ds = Dataset(X, y)
print("proporção da classe 0 (fora do círculo):", round(float(1 - y.mean()), 3))

# 3 learning rates x 3 tamanhos da camada escondida = 9 configurações, cada uma treinada em 5 folds
grid = {'learning_rate': [0.001, 0.01, 0.1], 'n_hidden': [2, 4, 16]}
results = grid_search_cv(build_network, ds, grid, cv=5, seed=1)
for h, s in zip(results['hyperparameters'], results['scores']):
    print(h, round(s, 3))
print("melhor:", results['best_hyperparameters'], round(results['best_score'], 3))
print("modelos treinados:", len(results['hyperparameters']) * 5)

# as configurações com 0.715: a rede prevê sempre a mesma classe?
for lr, n in [(0.001, 2), (0.001, 4)]:
    net = build_network(lr, n)
    net.fit(ds)
    print(f"lr {lr}, {n} neurónios -> previsões diferentes: {np.unique(np.round(net.predict(ds)))}")

proporção da classe 0 (fora do círculo): 0.715
{'learning_rate': 0.001, 'n_hidden': 2} 0.715
{'learning_rate': 0.001, 'n_hidden': 4} 0.715
{'learning_rate': 0.001, 'n_hidden': 16} 0.843
{'learning_rate': 0.01, 'n_hidden': 2} 0.725
{'learning_rate': 0.01, 'n_hidden': 4} 0.758
{'learning_rate': 0.01, 'n_hidden': 16} 0.975
{'learning_rate': 0.1, 'n_hidden': 2} 0.725
{'learning_rate': 0.1, 'n_hidden': 4} 0.75
{'learning_rate': 0.1, 'n_hidden': 16} 0.97
melhor: {'learning_rate': 0.01, 'n_hidden': 16} 0.975
modelos treinados: 45
lr 0.001, 2 neurónios -> previsões diferentes: [0.]
lr 0.001, 4 neurónios -> previsões diferentes: [0.]


In [5]:
# última pergunta do slide 12: passar uma rede já criada, em vez da função construtora
try:
    grid_search_cv(build_network(0.01, 16), ds, {'learning_rate': [0.0001, 0.1]}, cv=5, seed=1)
except AttributeError as erro:
    print("AttributeError:", erro)

# sem a verificação do hasattr, o setattr criava o atributo net.learning_rate, que ninguém lê:
# o optimizer foi criado no __init__ com 0.01 e copiado para as camadas no add
for lr in [0.0001, 0.1]:
    net = build_network(0.01, 16)
    net.learning_rate = lr                                   # o que o setattr faria
    scores = k_fold_cross_validation(net, ds, cv=5, seed=1)  # os mesmos folds do grid search
    print(f"net.learning_rate = {lr} | learning_rate do optimizer da 1.ª camada: "
          f"{net.layers[0].w_opt.learning_rate} | score: {np.mean(scores):.3f}")

# com a função construtora, o learning_rate muda mesmo
r = grid_search_cv(build_network, ds, {'learning_rate': [0.0001, 0.1], 'n_hidden': [16]}, cv=5, seed=1)
print("com a função construtora:", [round(s, 3) for s in r['scores']])

AttributeError: O modelo NeuralNetwork não tem o hiperparâmetro 'learning_rate'.
net.learning_rate = 0.0001 | learning_rate do optimizer da 1.ª camada: 0.01 | score: 0.975
net.learning_rate = 0.1 | learning_rate do optimizer da 1.ª camada: 0.01 | score: 0.975
com a função construtora: [0.715, 0.97]
